# CSC 391 Mini-Project 2: Naive Bayes Classification

In this assignment, you will be working on classifying messages for disaster aid using a `Naive Bayes` (`NB`) classifier. Victims of natural disasters have urgent needs for food, water, shelter, medicine, and other forms of aid. These needs are often communicated through text messages, social media posts, and local newspapers. Because of their ability to automatically process large amounts of text, `NLP` techniques can play an important role in ensuring that people receive potentially life-saving aid.
Our goal will be to perform text classification on messages sent in the aftermath of natural disasters.
After you are done testing your `NB` classifier on the disaster aid classification task, you will adapt it to another classification task: labelling the sentiment of messages related to `COVID` as positive or negative.

We will be utilizing a `Python` module called `NumPy` in this and the next assignment.
We are providing you with a `NumPy` tutorial (`numpy_tutorial.ipynb`) along with this assignment, which you can find in the same repository as this notebook. 
You can use the provided tutorial to learn `NumPy` for the first time or just refresh your knowledge.

## Roadmap




Read the entire assignment before proceeding.

There are `4` methods you need to implement in this assignment:
**`__init__()`**, **`train()`**, **`classify()`**, and **`get_vocab_probabilities()`** of the **[`NaiveBayesClassifier(Classifier)`]** class

Here is how your implementation will be evaluated:
* In `Evaluation on the Triage Dataset`, your 
  implementation will be evaluated with respect to the `triage` dataset.
  * you will check the accuracy of your `Naive Bayes` 
    model both on the `train` and `dev` sets, each with and without stop words.
    Hence, there will be a total of `4` accuracy tests in this section.
    We recommend going back to your implementation if the accuracies you get in 
    this section are far from what we have provided. 
* In `Evaluation on the COVID Dataset`, we will run the same evaluations
  as in `Part 4`, with the only difference being the dataset.

**Importing Modules.** Run the next cell to import the necessary modules we will use in this assignment.

In [1]:
""" Modules included in the Python Standard Library """

# collections module contain useful Python data structures, such as dictionaries
# with special properties
from collections import defaultdict

# operator module allows us to use functions such as add() instead of operators
# such as +
import operator

# random modules allows us to insert randomization to our code
import random

# typing module contains type objects. We will use these types to ensure that 
# the inputs and outputs passed to the functions you will be implementing are 
# of the correct type
from typing import List, Dict, Union
from collections import Counter
import math, os, random, re

In [2]:
""" Third party modules """

# numpy is a widely used scientific computing package, allowing us to do large
# matrix operations efficiently
import numpy as np

# matplotlib is a popular library used by researchers to plot graphs
import matplotlib.pyplot as plt

# sklearn is a popular machine learning library, providing useful tools for 
# machine learning tasks
from sklearn.feature_extraction.text import CountVectorizer

In [3]:
""" Our custom functions and classes """

# Helper functions and classes we will use later
from util import load_data, Classifier, Example, evaluate, remove_stop_words

**WARNING:** **DO NOT** import or use any other packages except the ones imported above and other packages in the Python standard library.
This means you should not use `spaCy`, `NLTK`, `gensim`, or other functionality in `scikit-learn` besides `CountVectorizer`, even though those are provided in the `conda` environment we set up for you.
If your solution uses any such extra dependencies it will fail the autograder.

## Part 1. Data Exploration for the Triage Dataset

As usual, the first thing to do is to understand and characterize the data!
The data for this assignment contains about `26K` documents from several major natural disasters, as listed below.

* [Earthquake in Haiti (2010)](https://en.wikipedia.org/wiki/2010_Haiti_earthquake)
* [Floods in Pakistan (2010)](https://en.wikipedia.org/wiki/2010_Pakistan_floods)
* [Earthquake in Chile (2010)](https://en.wikipedia.org/wiki/2010_Chile_earthquake)
* [Hurricane Sandy in North America (2012)](https://en.wikipedia.org/wiki/Hurricane_Sandy)

**Dataset.** The documents in our dataset are either text messages, social media (`Twitter`) posts, or snippets from news articles.
In addition to the specific events listed above the dataset contains a number of news articles spanning dozens of different disasters.
All messages have been translated and annotated by humans on the crowdsourcing platform `CrowdFlower` (now branded under [`Appen`](https://appen.com/)).
However, some of the translations are not perfect, and you may encounter some words in other
languages.
Unfortunately, `NLP` researchers often have to work with `messy` data.
If you are curious about the crowdsourcing translation effort for messages
from Haiti in particular, feel free to check out [this paper](https://nlp.stanford.edu/pubs/munro2010translation.pdf).

Your task is to classify each document as being aid-related, class `aid`, or not aid-related, class `not`.
Messages that are aid-related include individuals' requests for food, water, or shelter etc.
The `aid` class also includes news reports about dire situations and disaster relief efforts.
Below are several examples of aid-related documents, belonging to class `aid`.
```
Hello Good Morning We live on 31 Delmas we are without water without food and what we had have finished Please do something for us!
```
```
I am sending this SMS from Layah district for my sister whose house has got destroyed in a flood
So, the problem she faces now is that she hasn't got any 'Watan Card'or any financial aid from the government.
She has 5 children too.
```
```
Redcross came to my house and gave my family food ...
Guess were not getting power anytime soon . #sandy #RedCross
```
```
Relief officials have stressed the vital importance of bringing in clean drinking water and sanitation equipment to avoid deadly epidemics that in a
worst case scenario could claim as many or more lives than the tsunami itself.
```
Below are several examples of non-aid-related documents, belonging to class `not`:
```
A cold front is found over Cuba this morning.
It could cross Haiti tomorrow.
Isolated rain showers are expected over our region tonight.
```
```
Hurricane : A storm which New Yorkers use as an excuse to drink and eat junk
food in their pajamas for 48 hours . #sandy
```
```
By secret ballot, the Council elected Pakistan, Bahrain and the Republic of
Korea from the Asian States, while Iran and Saudi Arabia did not receive enough
votes to qualify.
```


**Training, Validation, and Test Sets.**
The data is divided into a `training` set, `development` (`validation`) set, and `test` set.
Recall that the `training` set is used to learn, compute the statistics
for, your model.
These statistics are then used to classify the documents in the
`development` and `test` sets.
For this assignment, you have access to the
`training` set and the `dev` set.
The test `set` is hidden, but your submission will be evaluated on it as well.

**Exploration.**
Let's take a look at some of the data.
We have defined a `Dataset` class for you to store the loaded data in a way we can easily access later, and a function `load_data()` to load it in the format we want.
You do not need to check the specifics of our `Dataset` class, we will explain exactly how you will use it in this assignment.

In [4]:
# Load our dataset
dataset = load_data("./data/triage")

# Check that the type of our dataset is the Dataset class we defined for you
# in util.py.
print(type(dataset))

<class 'util.Dataset'>


We are interested in the following two fields of the `Dataset` class: `train` and `dev`.
Given that `dataset` is an instance of the `Dataset` class, we can access these fields with `dataset.train` and `dataset.dev`.

In [5]:
print(f"dataset.train contains {len(dataset.train)} examples")
print(type(dataset.train[0]))

dataset.train contains 21046 examples
<class 'util.Example'>


Each of `dataset.train` and `dataset.dev` is a list of `Example`'s.
Similar to `Dataset`, `Example` is a class we have defined to represent each data point we have.
The `Example` class has two fields we will be concerned with: `words` and `label`.
`words` field corresponds to the list of words making up the example.
`label` field corresponds to the label of the data point, which is an integer that can only take one of two values: `1` for `aid` and `0` for `not` aid.

In [6]:
print("First training example:")
print("Words: {}".format(dataset.train[0].words))
print("Label: {}".format(dataset.train[0].label))

First training example:
Words: ['visitors', 'over', 'the', 'years', 'would', 'notice', 'the', 'gradual', 'erosion', 'of', 'greenery', 'the', 'deforestation', 'the', 'growing', 'barrenness', 'of', 'the', 'landscape', 'around', 'villages']
Label: 0


In summary, we use the custom defined classes `Example` and `Dataset` to represent our dataset and data points in a nice format so that we can work with them easily.
This is achieved by the `load_data()` function we called earlier.
At a high level, when we pass it the path `./data/triage`, `load_data()` finds the `CSV` files located there. 
Within each of these `CSV` files, each line is a single example, consisting
of a document (`string`) and a corresponding label.
The function `load_data()` reads each line in the `CSV` files as a new `Example`.
It tokenizes each document it reads and sets the `words` field of the `Example` class to be a list of words.
You can check the `CSV` files located in `./data/triage` to see the original format of these files.

Note that the data you are given is already preprocessed; all punctuation has been removed, except hashtags and apostrophes, and all text has been converted to lowercase. You do not need to do any additional preprocessing for our task.


## Part 2. Naive Bayes

Now that we have our data set up, we can get started on implementing our `Naive Bayes` classifier!
To help you, are are sharing a skeleton to get you started.
Your job is to finish implementing the `NaiveBayesClassifier` class!

In [7]:
class NaiveBayesClassifier(Classifier):
    """
    TODO: Implement the Multinomial Naive Bayes classifier with add-1 smoothing
    (Laplace smoothing)
    """
    def __init__(self,
                 filter_stop_words=False):
        super().__init__(filter_stop_words)

        self.flag = filter_stop_words
    
        self.vocabulary = []
        self.vocabulary_dict = {}
        
        self.aid_words = []
        self.aid = {}
        
        self.nonaid_words = []
        self.nonaid = {}
        
        # self.classified_list = []
        
        self.aid_examples = 0
        self.nonaid_examples = 0
        
        pass


    def train(self, examples: List[Example]) -> None:
        """
        TODO: Implement a function that takes in a list of labeled
        Examples and trains the classifier.

        You can call the remove_stop_words function we imported earlier to
        remove the stop words. 
        * List of stop words can be accessed with self.stop_words. 
        * self.filter_stop_words is a flag that specifies whether the stop words
          should be removed.
        """
        
        for example in examples:
            if self.flag != False: 
                filtered_example = remove_stop_words(example.words, self.stop_words)
                self.vocabulary.extend(filtered_example)
            else:
                self.vocabulary.extend(example.words)
        
        self.vocabulary_dict = Counter(self.vocabulary) 
        for word in self.vocabulary_dict:
            self.vocabulary_dict[word] = 1
       
        for example in examples:
            if example.label == 1:
                if self.flag != False: 
                    filtered_example = remove_stop_words(example.words, self.stop_words)
                    self.aid_words.extend(filtered_example)
                    self.aid_examples += 1
                else:
                    self.aid_words.extend(example.words)
                    self.aid_examples += 1
    
        
    
        self.aid = Counter(self.aid_words)
        
        self.aid.update(self.vocabulary_dict)

        for word in self.aid:
            self.aid[word] = (1.0 * self.aid[word]) / (len(self.aid_words) + len(self.vocabulary_dict))

        for example in examples:
            if example.label == 0:
                if self.flag != False: 
                    filtered_example = remove_stop_words(example.words, self.stop_words)
                    self.nonaid_words.extend(filtered_example)
                    self.nonaid_examples += 1
                else:
                    self.nonaid_words.extend(example.words)
                    self.nonaid_examples += 1
       
        self.nonaid = Counter(self.nonaid_words)
        self.nonaid.update(self.vocabulary_dict)
        
        for word in self.nonaid:
            self.nonaid[word] = (1.0 * self.nonaid[word]) / (len(self.nonaid_words) + len(self.vocabulary_dict))
        
        pass


    def classify(self, examples: List[Example],
                 return_scores: bool = False) -> Union[List[int], List[float]]:
        """
        TODO: Implement a function that takes a list of Examples and predicts
        their labels using the learned classifier.

        If return_scores = True, return the score prob(label = 1 | example)
        for each example instead.
        """
       
        classified_list = []
        for example in examples:
            score_aid = 0
            score_nonaid = 0
            for word in example.words:
                if word in self.aid:
                    score_aid = score_aid + math.log(self.aid[word])  
                if word in self.nonaid:
                    score_nonaid = score_nonaid + math.log(self.nonaid[word])
            
            score_aid = score_aid + math.log(self.aid_examples/(self.aid_examples + self.nonaid_examples))
            score_nonaid = score_nonaid + math.log(self.nonaid_examples/(self.aid_examples + self.nonaid_examples))
            
            if return_scores != False:
                if (score_aid >= score_nonaid):
                    classified_list.append(score_aid)
                else:
                    classified_list.append(score_nonaid)
            else:
                if (score_aid >= score_nonaid):
                    classified_list.append(1)
                else:
                    classified_list.append(0)
        
        return classified_list
        pass
     

    def get_vocab_probabilities(self, label: int) -> Dict:
        """
        TODO: Implement a function to return a dictionary of a unigram model 
        given its label. In other words, the method returns a unigram model (dictionary)
        of class one if label is 1, a model of class 0, if label is 0.
        """
        # CODE START
        if label == 1:
            return self.aid
        else:
            return self.nonaid
        pass
     
    def remove_common_words(self, examples: List[Example]):
        words = []
        for example in examples:
            words.extend(example.words)
        words = words(counter)
        vocab_probs_positive = nb_classifier.get_vocab_probabilities(1)
        top_10_positive = sorted(vocab_probs_positive.items(),
                         key=operator.itemgetter(1), reverse=True)[:10]
        bottom_10_positive = sorted(vocab_probs_positive.items(),
                         key=operator.itemgetter(1))[:10]

The interface is very simple:

* **`train()`** takes in a list of training `Example`s and updates the classifier based on the data. Note that you will want to save some information into the classifier class.
* **`classify()`** takes a list of `Example`s, which will have labels, but
you should not use them, and return a corresponding list of predicted labels
(1 or 0) in the same order.
If `return_scores = True`, it will return the score
`prob(label = 1 | example)` for each example instead.
* **`get_vocab_probabilities(label)`** should return a dictionary of the unigram model given its label.

Beyond these, everything else is up to you!
You are free to add other helper methods or any other data structures and instance variables that you need.

__WARNING:__ **DO NOT** change the interface of the methods listed above, as these will be called directly when grading.

## Part 3. Tips

We do have some hints and suggestions that might help along
the way before we jump to the evaluation section.
It is totally possible to get a working solution without following all of these suggestions, so feel free to use or ignore them as you would like:

**Use log probabilities.** We strongly recommend computing the probabilities as log probabilities in your implementation.
Recall that your Naive Bayes prediction will be the argmax of a product of many probabilities (the prior probability $P(c)$ and a bunch of conditional probabilities $P(x | c)$ ).
Each of these can individually be small numbers, so if you multiply many of them together, they may rapidly approach zero and even get rounded to 0, which will make it difficult or impossible for you to compare the actual values accurately.
Instead, you can take the log of the probability, which will transform the
product into a sum of logs. These will avoid any such bad behavior.
And because log is a monotonically increasing function, if $log(x) > log(y)$ then $x > y$.
So when computing your argmax, you can just compare the log probabilities 
directly and never need to worry about the true probabilities!

**Keep track of the vocabulary.** For the purposes of implementing Laplace Smoothing (+1 smoothing), it may be
helpful to keep track of the vocabulary, the set of all words you've
seen in the training data, or at least its' size.
The Python `set()` structure may be useful here.

**You can use raw counts or computed probabilities.** There are a few different ways you can go about storing the information from
learning in the `NaiveBayesClassifier` object.
Ultimately, when classifying you will need to compute probabilities from the counts, but it's up to you whether you would like to store the raw counts or the computed probabilities.

**Defaultdict!** You may find Python's [defaultdict](https://docs.python.org/3/library/collections.html#collections.defaultdict) helpful in your implementation when counting.

**Consider the `remove_stop_words` flag.** Don't forget to implement your classifiers so they behave differently depending on if stop word filtering is enabled.
You can filter stop words using the `remove_stop_words` function in `util.py`.
If `filter_stop_words` is `True`, the `Classifier` will have a list of stop words stored in `self.stop_words`.

**In Python, assignment is by reference.** Remember that in `Python`, assignment is by reference, not by value, for
non-primitive types.
Or to put things more simply, when you're assigning an existing list or dict to a new variable, it does NOT make a copy.
It just gives a reference to the existing list or dict.

  ```
  a = [1, 2, 3]

  # This does NOT make a copy of a. b now points to the same list as a
  b = a

  b.append(4)

  # Prints "[1, 2, 3, 4]"
  print(a)

  # If you would like to make a copy of a list, you should do it explicitly
  b = a.copy()

  b.append(5)

  print(b) # Prints "[1, 2, 3, 4, 5]"
  print(a) # Prints "[1, 2, 3, 4]"
  ```

**Unknown words in test time.** Unknown words in the dev/test set that are not seen in your training set should be ignored in your Naive Bayes computations

**Code length.** Our reference implementation is just under 100 lines of code, including the
skeleton code.
It's quite possible that you can make a working implementation in fewer lines (or more lines, that's totally fine too).
But if your implementation is way longer than this, that might be a sign that you are over-complicating things.

## Part 4. Evaluation on the Triage Dataset

### Part 4.1. Accuracy

Once your implementation is ready, you can try evaluating it on the disaster aid classification dataset as shown below.
Our implementation achieves the following statistics, so if you are getting similar results that probably means that your implementation is working well!
```
Performance on Unigrams, no stopword removal:
Accuracy (train): 0.82946878266654
Accuracy (dev): 0.7329965021375826
Performance on Unigrams with stopword removal:
Accuracy (train): 0.8446735721752352
Accuracy (dev): 0.7306645938593082
```
You will test the accuracy achieved by your implementation of the `NaiveBayesClassifier` on `train`, `dev`, and `test` datasets, both with and without stop words.
If you aren't getting the performance you are expecting in the below cell, go back to your `train` and `classify` methods.
Your implementation of `get_vocab_probabilities` doesn't affect the outputs of the two cells immediately below.

In [8]:
# Load our dataset
dataset = load_data("./data/triage")

In [9]:
print("Performance on unigrams, no stopword removal:")
nb_classifier = NaiveBayesClassifier(filter_stop_words=False)
evaluate(nb_classifier, dataset)

print("Performance on unigrams with stopword removal:")
nb_classifier_swr = NaiveBayesClassifier(filter_stop_words=True)
evaluate(nb_classifier_swr, dataset)

Performance on unigrams, no stopword removal:
Accuracy (train): 0.82946878266654
Accuracy (dev): 0.7329965021375826
Performance on unigrams with stopword removal:
Accuracy (train): 0.8446735721752352
Accuracy (dev): 0.7306645938593082


### Part 4.2 Sanity Check

Once we've implemented and trained our model, it's often helpful to do some
investigating to confirm that it's behaving the way we expect.
For a Naive Bayes model, there are couple of different ways we can do this.

**Proper probability distribution**. Given a word, we know that the probabilities of each unigram model should add up to ~`1.0`.
The next two cells test this property for `k` examples that are randomly chosen from the training dataset and used to create unigram models.
You may notice that sometimes the sum of the probabilities isn't exactly `1.0`, such as `0.9999999999999999` or `1.0000000000000002`.
This is fine, and even expected, as computers have a limited number of bits to hold numbers, which can lead to an overflow when we are dealing with floating point operations.
Note that, every time you run the below cell, you will get a different set of examples.
If you are getting unexpected values when you run the below two cells, go back to your `get_vocab_probabilities` implementation.

In [10]:
def sanity_check_vocab_probabilities(classifier, dataset):
    sampled_data = random.sample(dataset.train, 30)
    classifier.train(sampled_data)
    labels = set(example.label for example in sampled_data)
    
    for label in labels:
        total_probability = 0.0
        vocabulary = classifier.get_vocab_probabilities(label)
        for word in vocabulary:
            total_probability = total_probability + vocabulary[word]
        if total_probability <= 0.999 or total_probability >= 1.001:
            raise ValueError(f'The probabilities add up to {prob}')
        print(total_probability)

In [11]:
# Call sanity check
dataset = load_data("./data/triage")
nb_classifier = NaiveBayesClassifier(filter_stop_words=False)
sanity_check_vocab_probabilities(nb_classifier, dataset)

0.9999999999999938
0.9999999999999997


**Vocabulary Probabilities.**
Another good sanity check is to examine the learned conditional probabilities for each of the words in the vocabulary.
Specifically, we want to find the words for which the probability of a particular label is high.
For example, to find the words that the model thinks best indicate an aid-related message, we would want to find words with a high value of $P(\text{label} = 1 | \text{word})$.
We can do this easily using the `get_vocab_probabilities()` method.
Let's print out the top `10` highest-probability words for each labe, `aid` and `not`.

What did you think of the words you saw? 
Do they seem plausible to you? 

In [12]:
# Words that best indicate `aid`
dataset = load_data("./data/triage")
nb_classifier = NaiveBayesClassifier(filter_stop_words=True)
nb_classifier.train(dataset.train)

vocab_probs_positive = nb_classifier.get_vocab_probabilities(1)
top_10_positive = sorted(vocab_probs_positive.items(),
                         key=operator.itemgetter(1), reverse=True)[:10]
bottom_10_positive = sorted(vocab_probs_positive.items(),
                         key=operator.itemgetter(1))[:10]

for word, prob in top_10_positive:
    print("{}: prob = {}".format(word, prob))

print("\n")
    
for word, prob in bottom_10_positive:
    print("{}: prob = {}".format(word, prob))

food: prob = 0.010832698404895912
people: prob = 0.009898739155012657
water: prob = 0.009861872342517265
000: prob = 0.004577629218177797
earthquake: prob = 0.00436871728070391
relief: prob = 0.003059945437117507
aid: prob = 0.0030046452183744195
areas: prob = 0.0027097107184112863
affected: prob = 0.0025376655934327918
tents: prob = 0.002420920687197385


parent: prob = 6.144468749231942e-06
nrcs: prob = 6.144468749231942e-06
binou: prob = 6.144468749231942e-06
hike: prob = 6.144468749231942e-06
managucha: prob = 6.144468749231942e-06
gucha: prob = 6.144468749231942e-06
timberline: prob = 6.144468749231942e-06
briefings: prob = 6.144468749231942e-06
graphic: prob = 6.144468749231942e-06
renorunks: prob = 6.144468749231942e-06


In [13]:
def answer1():
    answer = "The words seem plausible. Since the top ten words include necessity such as food/ water, and disaster reated words as well. The one word in top 10 that deos not amke sense is 000, which we probably could parse out in the future. The bottom words make sense as well because these are rare ramdom words, or maybe in a different language."
    return answer
answer1()

'The words seem plausible. Since the top ten words include necessity such as food/ water, and disaster reated words as well. The one word in top 10 that deos not amke sense is 000, which we probably could parse out in the future. The bottom words make sense as well because these are rare ramdom words, or maybe in a different language.'

In [14]:
# Words that best indicate not `aid`
vocab_probs_negative = nb_classifier.get_vocab_probabilities(0)
top_10_negative = sorted(vocab_probs_negative.items(),
                         key=operator.itemgetter(1), reverse=True)[:10]

for word, prob in top_10_negative:
    print("{}: prob = {}".format(word, prob))

people: prob = 0.0045886614479506
water: prob = 0.004509651383283901
earthquake: prob = 0.004339475859386396
http: prob = 0.004272621189283804
information: prob = 0.0033123450187193077
haiti: prob = 0.0032090241649243936
food: prob = 0.002868673117129382
good: prob = 0.0024493120046676716
find: prob = 0.0024067681236932953
country: prob = 0.0024006904264112414


**False Positives and Negatives.** Another good thing to check is where your model made errors. In this case, our
task was binary classification, so there are two possible types of errors
that could have occurred:

* `False Positives`: Our model predicted a high probability of label = 1 for a negative example.
* `False Negatives`: Our model predicted a low probability of label = 1 for a positive example.

We can look for exactly these two types of errors using the `return_scores`
flag we asked you to implement for the `classify()` method.

In [15]:
def get_false_negatives_and_false_positives(classifier, examples):
    predicted_scores = classifier.classify(examples, return_scores=True)

    false_negatives = []
    false_positives = []

    for pred_score, example in zip(
            predicted_scores, examples):
        if example.label == 1 and pred_score < 0.5:
            false_negatives.append((example.words, pred_score))
        elif example.label == 0 and pred_score >= 0.5:
            false_positives.append((example.words, pred_score))

    return false_negatives, false_positives

fn, fp = get_false_negatives_and_false_positives(nb_classifier, dataset.dev)

Now that we have the false negatives and false positives, we can find the
"worst" ones and examine them to try to figure out where our model went wrong.

The **worst** ones would be:
* The `false negatives` with the lowest probabilities of label = 1
* The `false positives` with the highest probabilities of label = 1

Run the cells below, and think about the following questions:
* Do the mistakes you see seem reasonable/sensible? Why do you think the classifier
misclassified them?
* Could some of these misclassifications be avoided if we had access to more
training data? Or do some of them stem from the limitations of the Naive Bayes
model itself (Hint: Think about what assumptions go into the Naive Bayes model)?

In [16]:
def answer2():
    answer = "These mistakes mostly make sense because they make use of words that can be regarded as aid related; for example, words like \"catastrophic, earthquake\" would be more classified into aid than nonaid but yet it's describing an issue from the past years from the context. I think it might be hard to avoid because a lot of the times we need to look at the overall context instead of unigram model, which is a problem of Naive Bayes."
    return answer
answer2()

'These mistakes mostly make sense because they make use of words that can be regarded as aid related; for example, words like "catastrophic, earthquake" would be more classified into aid than nonaid but yet it\'s describing an issue from the past years from the context. I think it might be hard to avoid because a lot of the times we need to look at the overall context instead of unigram model, which is a problem of Naive Bayes.'

In [17]:
top_10_fn = sorted(fn,
                   key=operator.itemgetter(1))[:10]
for words, prob in top_10_fn:
    print("prob = {}: {}...".format(prob, words[:min(len(words), 10)]))

prob = -5352.656229015475: ['we', 'used', 'to', 'harvest', '15', 'carts', 'full', 'of', 'manioc', 'a']...
prob = -4951.604638365272: ['one', 'year', 'ago', 'when', 'a', 'catastrophic', 'earthquake', 'hit', 'china', 'canadians']...
prob = -4870.175291164863: ['emerging', 'viral', 'diseases', 'such', 'as', 'ebola', 'marburg', 'hemorrhagic', 'fever', 'and']...
prob = -3765.711579434267: ['since', '22', 'november', '2008', 'there', 'has', 'been', '719', '4', 'mm']...
prob = -3145.792990385084: ['the', 'country', 'has', 'been', 'devestated', 'by', 'two', 'decades', 'of', 'conflict']...
prob = -3101.02900269561: ['we', 'are', 'cooped', 'up', 'all', 'day', 'here', 'tarpaulin', 'sheets', 'will']...
prob = -2187.5106443239592: ['putih', 'river', 'and', 'pabelan', 'river', 'severe', 'overflow', 'carrying', 'mud', 'and']...
prob = -1921.6740045058766: ['diets', 'for', 'households', 'with', 'poor', 'and', 'limited', 'consumption', 'lack', 'animal']...
prob = -1376.3950066949747: ['we', 'are', 'afr

In [18]:
if len(fp) == 0:
    print("No false positives found!")

top_10_fp = sorted(fp, key=operator.itemgetter(1), reverse=True)[:10]
for words, prob in top_10_fp:
    print("prob = {}: {}".format(prob, words))

No false positives found!


Hopefully these questions have gotten you thinking about what your model
is doing and what its weaknesses and problems might be.
If you have time, we encourage you to spend some more time playing around with your model in this section.
Finally, when computing your model's performances with the different settings, no stop word removal vs. stop word removal, you may have
noticed something surprising.
In general, we would expect models using stop word removal to outperform those that don't in many cases.
However, you may have found that on this dataset, this does not necessarily
occur!
This does not mean that your implementation is broken or incorrect, although you should definitely double-check just to be sure.
Why do you think this might be happening? What could be changed to get the
expected behavior?

**NOTE:** It may be helpful to consider the differences between the training
set and dev accuracies. You can think back to our discussion of overfitting in the group work.

<a id="evaluation_covid"></a>
## Part 5. Evaluation on the COVID Dataset

Now that we have verified that our `Naive Bayes` model behave correctly on a given dataset, it is easy to apply it and evaluate on an alternative dataset and see if it perform differently.
In `data/coronavirus`, we have provided a second dataset consisting of `reddit` comments on posts related to the `COVID-19` pandemic early last year.
Each example is a single line in the `CSV` file, consisting of a comment and associated sentiment label: `0` for `negative`, and `1` for `positive`.
This dataset consists of around `4` times as many examples as the `triage/disaster` dataset.
The task on this dataset is to, given the text of a comment, predict its
sentiment label.
This is an example of sentiment analysis, specifically sentiment classification, which another type of `NLP` task.

In this particular case, you could imagine using sentiment analysis tools
to get an approximate idea of the mood social media, i.e. `Reddit`, users feel
about the pandemic at a particular point in time.
If we succeeded in training a good classifier to identify `positive` and `negative` sentiment comments/posts, we could use it to count the number of `positive` and `negative` posts each day, giving an approximate measure of social media sentiment.
This information could be very useful for governments or NGOs trying to gauge
the public response to `COVID-19` policies, or identify how the pandemic is
affecting mental health.
Although the task is different, we can straightforwardly load and examine the new data the same way we did with the previous dataset.

In [19]:
# Load our dataset
covid_dataset = load_data("./data/coronavirus")
print(type(covid_dataset))

print("dataset.train contains {} examples".format(len(covid_dataset.train)))

print("First training example:")
print("Words: {}".format(covid_dataset.train[0].words))
print("Label: {}".format(covid_dataset.train[0].label))

<class 'util.Dataset'>
dataset.train contains 80000 examples
First training example:
Words: ['It', 'gets', 'it', 'from', 'the', 'GEOS5', 'model', 'The', 'GEOS5', 'model', 'does', 'not', 'get', 'live', 'information', 'on', 'sulphur', 'emissions', 'from', 'satellite', 'data', 'Instead', 'it', 'combines', 'expected', 'sulphur', 'emissions', 'as', 'determined', 'from', 'data', 'from', 'the', '90s', 'and', '00s', 'with', 'weather', 'data', 'to', 'forecast', 'where', 'SO2', 'will', 'go', 'once', 'its', 'already', 'emitted']
Label: 1


We can train and evaluate our models just like before as well. 
Our solution achieves the following statistics when we run the below cell, so if you are getting similar results that probably means that your implementation is working well!
```
Performance on unigrams, no stopword removal:
Accuracy (train): 0.8691
Accuracy (dev): 0.7853
Performance on unigrams with stopword removal:
Accuracy (train): 0.8632125
Accuracy (dev): 0.7706
```
**Food for thougth:** What do you notice about the results you got on this dataset compared to the previous one?

In [20]:
def answer3():
    answer = "I noticed that our performance the train accuracy is about the same as the triage dataset and this is probably just a simple reflection of our training model algorithm. The accuracy for dev is overall higher regardless of stopword so the classify works better for this dataset than the previous one."
    return answer
answer3()

'I noticed that our performance the train accuracy is about the same as the triage dataset and this is probably just a simple reflection of our training model algorithm. The accuracy for dev is overall higher regardless of stopword so the classify works better for this dataset than the previous one.'

In [21]:
print("Performance on unigrams, no stopword removal:")
nb_classifier = NaiveBayesClassifier(filter_stop_words=False)
evaluate(nb_classifier, covid_dataset)

print("Performance on unigrams with stopword removal:")
nb_classifier_swr = NaiveBayesClassifier(filter_stop_words=True)
evaluate(nb_classifier_swr, covid_dataset)

Performance on unigrams, no stopword removal:
Accuracy (train): 0.8691
Accuracy (dev): 0.7853
Performance on unigrams with stopword removal:
Accuracy (train): 0.8632125
Accuracy (dev): 0.7706


Before we wrap up, let's run the quick probability sanity check our method `get_vocab_probabilities` using the `COVID` dataset.

In [22]:
sanity_check_vocab_probabilities(nb_classifier, covid_dataset)

1.0000000000025844
0.9999999999981898


## Submission

Congratulations, you are done with the assignment! Submit your mini_project2_updated.ipynb notebook in Canvas.